# NPU ResNet-18 acceptance demo

This Phase 2B notebook validates the packaged overlay and ResNet-18 acceptance assets, runs the corpus twice through the public NPU runtime, and reports accuracy, latency, work, and physical-cycle metrics. Run it from the extracted standalone package root on the PYNQ-Z1.

In [ ]:
from pathlib import Path
import sys

package_root = Path.cwd().resolve()
if not (package_root / 'package.manifest.json').is_file():
    raise RuntimeError('start this notebook from the extracted ResNet-18 package root')
for import_root in (package_root, package_root.parent):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from src.model.resnet18 import load_acceptance_bundle, validate_resnet18_topology
from src.runtime import NPUModelRuntime, load_model_package, load_pynq_runtime
from src.runtime.acceptance import run_resnet18_acceptance
from src.runtime.verify_overlay import verify_artifacts

In [ ]:
overlay = verify_artifacts(package_root / 'artifacts')
descriptor_path = package_root / 'acceptance' / 'acceptance.json'
initial_bundle = load_acceptance_bundle(descriptor_path)
model = load_model_package(initial_bundle.model_manifest_path)
bundle = load_acceptance_bundle(descriptor_path, graph=model.graph)
validate_resnet18_topology(model.graph)
print({
    'source_commit': overlay['source_commit'],
    'target_part': overlay['target_part'],
    'samples': bundle.descriptor.sample_count,
})

In [ ]:
physical = load_pynq_runtime(package_root / 'artifacts' / 'npu_matrix.bit')
runtime = NPUModelRuntime(physical, model)
print({
    'abi_major': physical.abi_major,
    'capabilities': physical.capabilities,
    'physical_limits': (physical.max_m, physical.max_n, physical.max_k),
})

In [ ]:
evidence = run_resnet18_acceptance(
    bundle, runtime, mode='board', repeat_count=2
)
assert evidence['gates']['repeatability'] is True
assert evidence['gates']['exact_output_ratio'] >= bundle.descriptor.thresholds.exact_output_min
assert evidence['gates']['top1_accuracy'] >= bundle.descriptor.thresholds.top1_min
assert evidence['performance']['physical_cycles'] is not None
print({
    'exact_output_ratio': evidence['gates']['exact_output_ratio'],
    'top1_accuracy': evidence['gates']['top1_accuracy'],
    'invocations': evidence['performance']['invocations'],
    'latency_ns': evidence['performance']['latency_ns'],
    'physical_jobs': evidence['performance']['physical_jobs'],
    'physical_cycles': evidence['performance']['physical_cycles'],
})
print('PASS: NPU ResNet-18 notebook demo')